Attempting to apply Gauss Seidel to $x_0=\sin(\pi x)\sin(\pi y)+\frac{1}{10}\sin(10\pi x)\sin(10\pi y)$  
Then attempting to create two levels of finite element spaces such that one is a coarser version of the other and make a prolongation P matrix which projects from one space to the other.  
Overall goal is to:  
1. apply a custom number of GS smoothing steps to our initial iterate $x_0$,  
2. visualize the current iterate after such smoothing,   
3. then project onto the coarse level,   
4. display the iterate in its coarse form,   
5. direct solve,   
6. then project back to the fine level and   
7. view the solution.

In [3]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np

In [4]:
coarse_mesh = Mesh(unit_square.GenerateMesh(maxh=1/16))
fesc = H1(coarse_mesh, order=1, dirichlet="left|right")
fine_mesh = Mesh(coarse_mesh.ngmesh.Copy())
fesf = H1(fine_mesh, order=1, dirichlet="left|right")

Draw(coarse_mesh)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [5]:
fesf.mesh.Refine()

Draw(fesf.mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [6]:
Draw(fesc.mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [12]:
P = fesf.Prolongation().CreateMatrix(fesf.mesh.levels-1)
PT = P.CreateTranspose()
PT.shape
#print(PT)

(339, 1289)

In [8]:
u, v = fesf.TnT()
a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
f = LinearForm(v*dx).Assemble()
print(a.mat)


Row 0:   0: 1   339: -4.17957e-15   730: -0.5   1211: -0.5
Row 1:   1: 0.851839   361: -0.243593   751: -0.304714   753: -0.303532
Row 2:   2: 1   376: 0   782: -0.5   1218: -0.5
Row 3:   3: 0.82844   396: -0.413508   805: -0.205344   807: -0.209588
Row 4:   4: 2.05806   339: -0.15937   340: 0.161458   730: -0.5   731: -0.685102   832: -0.875045
Row 5:   5: 2.33332   340: -0.198708   341: 0.454755   731: -0.523643   732: -0.791008   733: -1.27472
Row 6:   6: 1.74449   341: -0.464606   342: -0.279878   732: -0.336252   734: -0.663759
Row 7:   7: 2.26331   342: -0.279878   343: -0.266442   344: 0.530118   735: -1.14298   1212: -1.10413
Row 8:   8: 2.27262   343: -0.266442   345: -0.266451   346: 0.539247   736: -1.14059   1213: -1.13838
Row 9:   9: 2.27741   345: -0.266451   347: -0.266888   348: 0.543966   737: -1.15374   1214: -1.1343
Row 10:   10: 2.27887   347: -0.266888   349: -0.265022   350: 0.544867   738: -1.17057   1215: -1.12125
Row 11:   11: 2.27561   349: -0.265022   351: -0

In [75]:
a_fine = a.GetMatrixLevel()
#a_fine.shape
m = a_fine.CreateSmoother(fesf.FreeDofs(), GS=True)


In [ ]:
# From GS Smoother in 2.1.2 of iTutorials
# haven't tested this cell yet, needs editing, but wanted to save it
gfu.vec[:] = 0
res = f.vec.CreateVector()              # residual
projres = f.vec.CreateVector()          # residual projected to freedofs
proj = Projector(fes.FreeDofs(), True)

for i in range(500):
    preJpoint.Smooth(gfu.vec, f.vec)    # one step of point Gauss-Seidel
    res.data = f.vec - a.mat*gfu.vec
    projres.data = proj * res
    print ("it#", i, ", res =", Norm(projres))
Draw (gfu)

In [ ]:
x0 = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/10)*sin(10*pi*x)*sin(10*pi*y))
ksteps = 5
xi = GridFunction(fesf, multidim=ksteps)
xi.Set(x0)
Draw(xi,fine_mesh,deformation=True)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [71]:
yi = GridFunction(fesf)
ainv = a_fine.Inverse()
iish = a_fine @ ainv
help(iish)
# xi.vec.data -= ainv * xi.vec
# Draw(xi, fine_mesh, Deformation=True)

Help on ProductMatrix in module ngsolve.la object:

class ProductMatrix(BaseMatrix)
 |  Method resolution order:
 |      ProductMatrix
 |      BaseMatrix
 |      pybind11_builtins.pybind11_object
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, /, *args, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  ----------------------------------------------------------------------
 |  Readonly properties defined here:
 |  
 |  matA
 |  
 |  matB
 |  
 |  ----------------------------------------------------------------------
 |  Methods inherited from BaseMatrix:
 |  
 |  AsVector(...)
 |      AsVector(self: ngsolve.la.BaseMatrix) -> ngsolve.la.BaseVector
 |      
 |      Interprets the matrix values as a vector
 |  
 |  CreateColVector(...)
 |      CreateColVector(self: ngsolve.la.BaseMatrix) -> ngsolve.la.BaseVector
 |  
 |  CreateDeviceMatrix(...)
 |      CreateDeviceMatrix(self: ngsolve.la.BaseMatrix) -> BaseMatrix
 |  
 

In [36]:
zi = GridFunction(fesf)
smootha = m @ a_fine
for i in range(3):
    smootha.Mult(yi.vec, zi.vec.data)
    yi.vec.data = zi.vec.data
    Draw(yi,fine_mesh,deformation=True)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [ ]:
a_coarse = PT @ a_fine @ P
#a_coarse.shape

print(a_coarse)

Help on SymmetricGaussSeidelPreconditioner in module ngsolve.la object:

class SymmetricGaussSeidelPreconditioner(BaseMatrix)
 |  Method resolution order:
 |      SymmetricGaussSeidelPreconditioner
 |      BaseMatrix
 |      pybind11_builtins.pybind11_object
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, /, *args, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  ----------------------------------------------------------------------
 |  Methods inherited from BaseMatrix:
 |  
 |  AsVector(...)
 |      AsVector(self: ngsolve.la.BaseMatrix) -> ngsolve.la.BaseVector
 |      
 |      Interprets the matrix values as a vector
 |  
 |  CreateColVector(...)
 |      CreateColVector(self: ngsolve.la.BaseMatrix) -> ngsolve.la.BaseVector
 |  
 |  CreateDeviceMatrix(...)
 |      CreateDeviceMatrix(self: ngsolve.la.BaseMatrix) -> BaseMatrix
 |  
 |  CreateMatrix(...)
 |      CreateMatrix(self: ngsolve.la.BaseMatrix) -> BaseMatri

In [10]:
gfu = GridFunction(fes1)
s = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/10)*sin(10*pi*x)*sin(10*pi*y))
gfu.Set(s)
Draw(gfu, deformation=True)

NameError: name 'fes1' is not defined

In [ ]:
)
# u, v = fesf.TnT()
# a = BilinearForm(grad(u)*grad(v)*dx)
# f = LinearForm(v*dx)
# gfu = GridFunction(fes1)
# s = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/10)*sin(10*pi*x)*sin(10*pi*y))
# gfu.Set(s)
# Draw(gfu, deformation=True)